# Phase 3 — Source-Held-Out Probes

**The question.** Does the model read clauses, or does it recognise datasets? The fused
corpus stitches three annotation projects together, each with its own drafting register,
clause segmentation and label vocabulary. A model that has learned "this looks like a
CLAUDETTE row, and CLAUDETTE rows about arbitration are usually harmful" would score well
on a random split while having learned very little about arbitration.

The existing `notebooks/model_finetuning/lawgic_classifier_probe.ipynb` already showed
that source identity is *linearly decodable* from the fine-tuned encoder. That is
necessary but not sufficient evidence: an encoder can carry source information without the
heads depending on it. This notebook tests the stronger claim directly — **remove a source
from training entirely, then evaluate only on that source's rows.**

## Two probes, and why not three

| Probe | Held out | Corpus rows carrying that source |
| --- | --- | --- |
| A | CLAUDETTE | 3,182 wide rows (3,721 long-format annotation rows) |
| B | 100 ToS | 1,460 wide rows (2,048 long-format annotation rows) |
| — | ~~ToS;DR~~ | **deliberately not run** |

The row counts differ between the long and wide formats because the wide corpus is one row
per unique clause: a clause annotated with several topics by the same source collapses into
one row with several active topic cells. The holdout operates on wide rows, so those are
the numbers reported.

**Why there is no ToS;DR holdout.** ToS;DR supplies 21,949 of 26,479 rows — about 83% of
the corpus and roughly 88% of training rows once the split is applied. Removing it would
leave ~2,500 training clauses. A score collapse under that condition is uninterpretable:
it would be perfectly consistent with "the model only recognised ToS;DR" *and* with "no
model learns 44-way multi-label legal topic detection from 2,500 examples". The probe
would be confounded with data starvation and would answer neither question. The two
smaller sources can be removed while leaving the training regime broadly intact, which is
what makes their results readable.

## Protocol

Legal-BERT, seed 42, dual-head, identical to Phase 2 in every other respect — same
persisted seed-42 split, same hyperparameters, same losses, same early stopping, same
degenerate-model assertion. The holdout removes the source's rows from **train and
validation** and restricts the **test** set to exactly those rows.

## Masking: score only what the held-out source actually supervised

This is the part that decides whether the probe means anything.

The supervision mask is source-aware: a row annotated by CLAUDETTE has observed cells only
for the topics CLAUDETTE's label vocabulary covers; every other cell is *unknown* and
contributes zero loss. If a held-out CLAUDETTE row is scored across all 44 topics, most of
the score comes from cells CLAUDETTE never labelled — the model is being graded against
the shape of the mask, not against comprehension of the clause.

`source_supervision_mask()` in `scripts/lawgic_train_matrix.py` rebuilds, per row, the set
of topic cells the held-out source itself asserted (read back out of `native_annotations`,
which records `source_dataset` and `lawgic_topic_id` for every annotation) and intersects
it with the row's existing supervision mask. Scoring runs over that intersection only.

The in-distribution baseline is computed on **exactly the same rows and the same cell
mask**, taken from the Phase 2 legal-bert/seed-42 run's stored logits. Without that
restriction the "retained ratio" would be comparing two different denominators and would
be meaningless.

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    sentinel = Path("generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv")
    for candidate in (start, *start.parents):
        if (candidate / sentinel).exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json

import numpy as np
import pandas as pd

import lawgic_eval_core as core
import lawgic_train_matrix as tm

pd.set_option("display.width", 160)

core.persist_split()
corpus = core.load_corpus()
frames = core.split_frames(corpus)
assert {k: len(v) for k, v in frames.items()} == core.EXPECTED_SPLIT_ROWS

HOLDOUT_SOURCES = ["claudette", "100_tos"]
BASELINE_RUN = tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=42, heads="dual").run_id

for source in HOLDOUT_SOURCES + ["tos_dr"]:
    corpus_rows = int(tm.source_row_mask(corpus, source).sum())
    train_rows = int(tm.source_row_mask(frames["train"], source).sum())
    test_rows = int(tm.source_row_mask(frames["test"], source).sum())
    print(f"{source:>10}: corpus {corpus_rows:>6,} | train {train_rows:>6,} "
          f"({train_rows / len(frames['train']):.1%}) | test {test_rows:>5,}")

print(f"\nPhase 2 baseline run required: {BASELINE_RUN}")

## MANUAL STEP — prerequisites

1. **Run Phase 2 first.** This notebook reads
   `generated_files/lawgic_taxonomy/runs/legal-bert-base-uncased__seed42__dual/test_logits.npz`
   for the in-distribution baseline. Without it there is nothing to compare against.
2. **GPU.** Two more full fine-tunes, same order of magnitude per run as a Phase 2 run
   (slightly faster — training sets are smaller by the held-out source).
3. No downloads or credentials: legal-bert is already local.

In [ ]:
baseline_path = tm.RUNS_DIR / BASELINE_RUN / "test_logits.npz"
if not baseline_path.exists():
    raise FileNotFoundError(
        f"{baseline_path} missing. Run 02_multiseed_encoder_runs.ipynb (at least the "
        f"legal-bert/seed42/dual config) before this notebook."
    )
print(f"Baseline logits found: {baseline_path}")

## Run the two probes

Resumable in the same way as Phase 2: completed probes are skipped.

In [ ]:
FORCE_RERUN = False

probe_configs = [
    tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=42, heads="dual", holdout_source=source)
    for source in HOLDOUT_SOURCES
]

probe_records = []
for config in probe_configs:
    target = tm.RUNS_DIR / config.run_id / "metrics.json"
    if target.exists() and not FORCE_RERUN:
        print(f"skip {config.run_id} (already complete)")
        probe_records.append(json.loads(target.read_text()))
        continue
    print(f"running {config.run_id} ...")
    probe_records.append(tm.run_config(config))

display(pd.DataFrame(probe_records)[
    ["run_id", "train_rows", "val_rows", "test_rows", "wall_seconds", *core.HEADLINE_METRICS]
])

## In-distribution baseline on the same rows and same cells

The Phase 2 run scored the full 2,648-row test split. Here it is re-scored on the subset
of test rows belonging to the held-out source, under that source's own supervision mask —
the identical evaluation surface the probe faces.

In [ ]:
baseline = np.load(baseline_path)
baseline_row_ids = baseline["row_id"]
row_position = {int(r): i for i, r in enumerate(baseline_row_ids)}

test_frame = frames["test"]


def in_distribution_metrics(source: str) -> dict:
    """Phase 2 legal-bert/seed42 performance restricted to `source`'s test cells."""
    source_rows = test_frame[tm.source_row_mask(test_frame, source)].copy()
    positions = np.array([row_position[int(r)] for r in source_rows["row_id"]])

    arrays = core.label_arrays(source_rows)
    arrays["label_masks"] = tm.source_supervision_mask(source_rows, source)

    return core.all_metrics(
        baseline["topic_logits"][positions],
        baseline["harm_logits"][positions],
        arrays,
    )


rows = []
for source, record in zip(HOLDOUT_SOURCES, probe_records):
    indist = in_distribution_metrics(source)
    rows.append({"source": source, "condition": "in-distribution (Phase 2)", **indist})
    rows.append({"source": source, "condition": "held-out (Phase 3)",
                 **{k: record[k] for k in indist if k in record}})

comparison = pd.DataFrame(rows)
display(comparison)

## Retained-performance ratio

`held-out / in-distribution`, per metric. Read it as: **what fraction of its ability does
the model keep when it has never seen a single clause from this source?**

Interpretation guide for the manuscript — state the reading you adopt rather than letting
the number speak for itself:

- **near 1.0** — the model generalises across annotation projects; performance is not an
  artifact of source recognition.
- **materially below 1.0** — part of the headline score depends on having seen that
  source's register during training. That is a real limitation of the fused-corpus design,
  not necessarily a modelling failure.
- **The supervised-cell counts matter.** With few observed cells the ratio is noisy; the
  `observed_cells` column below is there so the ratio is never read without its
  denominator.

In [ ]:
metrics_for_ratio = list(core.HEADLINE_METRICS)

ratio_rows = []
for source in HOLDOUT_SOURCES:
    indist = comparison[(comparison["source"] == source) & (comparison["condition"].str.startswith("in-"))].iloc[0]
    held = comparison[(comparison["source"] == source) & (comparison["condition"].str.startswith("held"))].iloc[0]
    for metric in metrics_for_ratio:
        ratio_rows.append({
            "Source": source,
            "Metric": metric,
            "In-distribution": float(indist[metric]),
            "Held-out": float(held[metric]),
            "Retained ratio": float(held[metric]) / float(indist[metric]) if indist[metric] else float("nan"),
            "Test rows": int(held["rows"]),
            "Observed cells": int(indist["topic_observed_positions"]),
        })

probe_table = pd.DataFrame(ratio_rows)
display(probe_table)

core.write_outputs(
    probe_table,
    "phase3_source_holdout",
    caption=(
        "Source-held-out probes. Each probe retrains Legal-BERT (seed 42, dual-head, "
        "identical protocol) with one source removed from train and validation, then "
        "evaluates only on that source's test rows and only on the topic cells that "
        "source itself supervised. The in-distribution column is the Phase 2 "
        "legal-bert/seed-42 run scored on the identical rows and cells. No ToS;DR probe is "
        "reported: it would remove ~88\\% of training rows, confounding source recognition "
        "with data starvation."
    ),
    label="tab:source-holdout",
)

### Per-topic detail (optional)

Which topics survive the holdout and which collapse is usually more informative than the
aggregate. Topics with a handful of observed cells will swing wildly — read the `observed`
column before drawing any conclusion from a single row.

In [ ]:
for config in probe_configs:
    path = tm.RUNS_DIR / config.run_id / "per_topic.csv"
    if not path.exists():
        continue
    table = pd.read_csv(path)
    table = table[table["observed"] > 0].sort_values("f1", ascending=False)
    print(f"\n=== {config.run_id} ===")
    display(table.head(20))